### Database Schema Design

Our DuckDB database uses a multi-layered architecture to process data from ingestion to analysis:

1. **`raw` Schema**: 
   - **Table**: `raw.weather_daily`
   - **Purpose**: Direct append-only storage of the ingested Parquet files. It perfectly mirrors the raw extraction structure with no transformations.

2. **`staging` Schema**: 
   - **Table**: `staging.weather_daily` (To be built next)
   - **Purpose**: The cleaned version of raw data. This layer enforces strict data types, handles null values (like the 7-day forecast gaps in soil moisture), removes duplicates, and standardizes formats.

3. **`analytics` Schema**:
   - **Table**: `analytics.city_weather_features` (To be built next)
   - **Purpose**: Enriched, aggregated tables built specifically for downstream ML inference and BI reporting. Contains rolling averages, lag features, and seasonal indicators.

In [14]:
import sys
import os
import pandas as pd

# Add the src folder to Python's path so we can import our new module
sys.path.append(os.path.abspath("../src"))
import database as db

# Initialize Connection
db_path = "../data/weather_pipeline.duckdb"
conn = db.get_connection(db_path)

# Execute DB Setup Tasks
db.create_schemas(conn)
db.create_raw_tables(conn)
db.load_raw_data(conn, "../data/raw/")

print("✅ Schemas created, tables defined, and raw data loaded successfully!")

✅ Schemas created, tables defined, and raw data loaded successfully!


In [15]:
print("--- 1. Row Counts per City ---")
display(conn.execute("""
    SELECT city, COUNT(*) as total_rows 
    FROM raw.weather_daily 
    GROUP BY city 
    ORDER BY total_rows DESC;
""").df())

print("\n--- 2. Date Range Verification ---")
display(conn.execute("""
    SELECT city, MIN(date) as start_date, MAX(date) as end_date 
    FROM raw.weather_daily 
    GROUP BY city;
""").df())

print("\n--- 3. Duplicate Checks ---")
df_dupes = conn.execute("""
    SELECT city, date, COUNT(*) as count 
    FROM raw.weather_daily 
    GROUP BY city, date 
    HAVING COUNT(*) > 1;
""").df()

if df_dupes.empty:
    print("✅ PASS: No duplicate date-city combinations found.")
else:
    print("❌ FAIL: Duplicates found!")
    display(df_dupes)

print("\n--- 4. Data Type Verification ---")
display(conn.execute("DESCRIBE raw.weather_daily;").df()[['column_name', 'column_type']])

--- 1. Row Counts per City ---


,city,total_rows
0,Lenkeran,17
1,Quba,17
2,Baki,17
3,Zerdab,17
4,Saatli,17



--- 2. Date Range Verification ---


,city,start_date,end_date
0,Lenkeran,2026-04-14,2026-04-30
1,Quba,2026-04-14,2026-04-30
2,Saatli,2026-04-14,2026-04-30
3,Zerdab,2026-04-14,2026-04-30
4,Baki,2026-04-14,2026-04-30



--- 3. Duplicate Checks ---
✅ PASS: No duplicate date-city combinations found.

--- 4. Data Type Verification ---


,column_name,column_type
0,city,VARCHAR
1,date,TIMESTAMP
2,temperature_2m_mean,DOUBLE
3,et0_fao_evapotranspiration_sum,DOUBLE
4,sunshine_duration,DOUBLE
5,shortwave_radiation_sum,DOUBLE
6,relative_humidity_2m_mean,DOUBLE
7,surface_pressure_mean,DOUBLE
8,precipitation_sum,DOUBLE
9,precipitation_hours,DOUBLE


In [16]:
# 1. Average Temperature per city per year
print("--- Average Temperature per City per Year ---")
display(conn.execute("""
    SELECT 
        city, 
        EXTRACT(YEAR FROM date) as year, 
        ROUND(AVG(temperature_2m_mean), 2) as avg_temp 
    FROM raw.weather_daily 
    GROUP BY city, year 
    ORDER BY city, year;
""").df())

# 2. City with the highest variance in daily precipitation
print("\n--- Highest Variance in Daily Precipitation ---")
display(conn.execute("""
    SELECT 
        city, 
        ROUND(VARIANCE(precipitation_sum), 4) as precip_variance 
    FROM raw.weather_daily 
    GROUP BY city 
    ORDER BY precip_variance DESC 
    LIMIT 1;
""").df())

# 3. Top 10 hottest days across all cities
print("\n--- Top 10 Hottest Days ---")
display(conn.execute("""
    SELECT city, CAST(date AS DATE) as exact_date, temperature_2m_mean 
    FROM raw.weather_daily 
    ORDER BY temperature_2m_mean DESC NULLS LAST 
    LIMIT 10;
""").df())

# 4. How many days had zero precipitation per city per year?
print("\n--- Days with Zero Precipitation per City per Year ---")
display(conn.execute("""
    SELECT 
        city, 
        EXTRACT(YEAR FROM date) as year, 
        SUM(CASE WHEN precipitation_sum = 0 THEN 1 ELSE 0 END) as zero_precip_days 
    FROM raw.weather_daily 
    GROUP BY city, year 
    ORDER BY zero_precip_days DESC, city ASC;
""").df())

--- Average Temperature per City per Year ---


,city,year,avg_temp
0,Baki,2026,12.81
1,Lenkeran,2026,13.62
2,Quba,2026,7.95
3,Saatli,2026,14.17
4,Zerdab,2026,14.54



--- Highest Variance in Daily Precipitation ---


,city,precip_variance
0,Saatli,16.159



--- Top 10 Hottest Days ---


,city,exact_date,temperature_2m_mean
0,Zerdab,2026-04-27,17.952332
1,Saatli,2026-04-27,17.385168
2,Zerdab,2026-04-19,16.541666
3,Saatli,2026-04-19,16.497917
4,Zerdab,2026-04-23,16.418751
5,Zerdab,2026-04-30,16.089334
6,Saatli,2026-04-30,15.994416
7,Zerdab,2026-04-25,15.860668
8,Lenkeran,2026-04-20,15.713666
9,Zerdab,2026-04-20,15.597917



--- Days with Zero Precipitation per City per Year ---


,city,year,zero_precip_days
0,Zerdab,2026,11.0
1,Lenkeran,2026,10.0
2,Baki,2026,9.0
3,Saatli,2026,9.0
4,Quba,2026,5.0


In [17]:
display(conn.execute("""
    SELECT *
    FROM raw.weather_daily;
""").df())

,city,date,temperature_2m_mean,et0_fao_evapotranspiration_sum,sunshine_duration,shortwave_radiation_sum,relative_humidity_2m_mean,surface_pressure_mean,precipitation_sum,precipitation_hours,wind_speed_10m_max,cloud_cover_mean,wind_gusts_10m_mean,soil_moisture_0_to_7cm_mean,data_type
0,Baki,2026-04-14,11.312499,2.305472,33141.066406,15.010000,79.314529,1021.701904,2.4,13.0,15.856354,79.000000,22.964998,0.283333,historical
1,Baki,2026-04-15,11.702085,3.136395,41735.414062,20.240000,69.572823,1024.161987,0.3,3.0,15.484185,47.250000,28.199997,0.277208,historical
2,Baki,2026-04-16,13.097917,3.106966,42755.191406,19.440001,65.637993,1023.956787,0.0,0.0,12.362475,20.416666,14.085001,0.260625,historical
3,Baki,2026-04-17,12.995834,3.021810,42742.210938,21.790001,80.436111,1024.379028,0.0,0.0,16.036171,22.750000,25.874998,0.239250,historical
4,Baki,2026-04-18,13.668750,2.861732,43815.843750,22.580000,83.649406,1016.915588,0.0,0.0,22.350464,27.541666,38.280003,0.212958,historical
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,Zerdab,2026-04-26,14.039833,3.082464,29227.609375,17.500000,68.958336,1020.760742,0.0,0.0,19.576189,85.791664,17.445000,NaN,forecast
81,Zerdab,2026-04-27,17.952332,5.028692,45515.457031,23.530001,60.458332,1010.165833,0.0,0.0,24.633049,83.708336,23.984999,NaN,forecast
82,Zerdab,2026-04-28,15.491916,3.685580,16628.697266,10.940000,50.041668,1019.290955,0.0,0.0,26.980793,88.125000,35.595001,NaN,forecast
83,Zerdab,2026-04-29,12.838752,2.393121,16414.091797,13.720000,56.489582,1022.086243,0.0,0.0,6.193674,90.375000,8.827499,NaN,forecast


In [18]:
conn.close()